# Outlier Detection: Six Methods from Z-Scores to Variational Autoencoders

Companion notebook for the blog post *Outlier Detection: Six Methods from Z-Scores to Variational Autoencoders*.

**Dataset:** NASA Statlog Satellite — 5,100 multi-spectral sensor readings, 36 features, 1.47% anomaly rate.

**Methods covered:**
1. Z-Score
2. IQR Fencing
3. Mahalanobis Distance
4. k-NN Distance
5. Isolation Forest
6. One-Class SVM
7. Gaussian Mixture Model
8. Variational Autoencoder

All thresholds are set using a contamination prior (the expected outlier fraction). Labels are withheld until the final evaluation section.

---
## 1. Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy.stats import chi2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# 0=normal (inlier), 1=anomaly (outlier)
COLORS = {0: '#4C8FAF', 1: '#E07B72'}
LABELS = {0: 'Normal', 1: 'Anomaly'}

os.makedirs('plots', exist_ok=True)

In [ ]:
# ── Load Satellite dataset and standardize ───────────────────────────────────

sat   = fetch_openml(data_id=40900, as_frame=False, parser='auto')
X_raw = sat.data          # (5100, 36)
y     = (sat.target == 'Anomaly').astype(int)   # 1=anomaly, 0=normal

scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

# y_out kept separate — used only in final evaluation
y_out = y.copy()

# Contamination prior: expected anomaly fraction
CONTAM = float(y_out.mean())

# 2-D PCA projection for scatter plots
pca2 = PCA(n_components=2, random_state=SEED)
X_2d = pca2.fit_transform(X)

c_vec = [COLORS[int(label)] for label in y]

print(f'Dataset : {X.shape[0]} samples, {X.shape[1]} features')
print(f'Anomaly (outlier): {y_out.sum()}  ({y_out.mean()*100:.2f}%)')
print(f'Normal  (inlier) : {(y_out==0).sum()}  ({(y_out==0).mean()*100:.2f}%)')
print(f'Contamination prior: {CONTAM:.4f}')

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def outlier_scatter(ax, X_2d, y_pred_out, title, flagged_color='red',
                    bg_colors=None, alpha=0.5, circle_flagged=True):
    """Scatter on PCA-2D axes. Flagged outliers (y_pred_out==1) circled in red."""
    if bg_colors is None:
        bg_colors = c_vec
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=bg_colors, s=8, alpha=alpha,
               linewidths=0, zorder=1)
    if circle_flagged:
        flagged = np.where(y_pred_out == 1)[0]
        ax.scatter(X_2d[flagged, 0], X_2d[flagged, 1],
                   facecolors='none', edgecolors=flagged_color,
                   s=30, linewidths=0.8, zorder=2, label=f'Flagged ({len(flagged)})')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('PC 1', fontsize=9)
    ax.set_ylabel('PC 2', fontsize=9)
    ax.legend(fontsize=8, frameon=False)


def compute_metrics(y_true_out, y_pred_out, scores):
    return {
        'Precision': precision_score(y_true_out, y_pred_out, zero_division=0),
        'Recall':    recall_score(y_true_out, y_pred_out, zero_division=0),
        'F1':        f1_score(y_true_out, y_pred_out, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_true_out, scores),
    }


def contam_threshold(scores, contam=None):
    """Flag the top `contam` fraction of scores as outliers."""
    if contam is None:
        contam = CONTAM
    return float(np.percentile(scores, (1 - contam) * 100))


print('Helpers defined.')

---
## 2. Statistical Thresholds

### Z-Score

After standardizing, each feature has mean 0 and standard deviation 1. We flag a sample if **any** feature exceeds threshold $\tau$ in absolute value:

$$\hat{y}_i = 1 \iff \max_j |z_{ij}| > \tau$$

### IQR Fencing

Tukey's fence defines per-feature bounds $Q_{1,j} - k \cdot \text{IQR}_j$ and $Q_{3,j} + k \cdot \text{IQR}_j$. A sample is flagged if it falls outside the fence on at least one feature.

### Mahalanobis Distance

Accounts for the full covariance structure:

$$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

Under multivariate normality, $D_M^2 \sim \chi^2_p$. We use the pseudo-inverse to handle correlated features.

In [ ]:
# ── Z-score and IQR threshold sweeps ─────────────────────────────────────────────

z_scores = np.abs(X)
max_z    = z_scores.max(axis=1)

tau_vals       = [2.0, 2.5, 3.0, 3.5, 4.0]
zscore_flagged = [(max_z > tau).mean() * 100 for tau in tau_vals]

Q1  = np.percentile(X, 25, axis=0)
Q3  = np.percentile(X, 75, axis=0)
IQR = Q3 - Q1

k_vals      = [1.0, 1.5, 2.0, 3.0]
iqr_flagged = []
for k in k_vals:
    outside = ((X < Q1 - k * IQR) | (X > Q3 + k * IQR)).any(axis=1)
    iqr_flagged.append(outside.mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(tau_vals, zscore_flagged, marker='o', color='#4C8FAF', linewidth=2)
ax.axhline(CONTAM * 100, color='#E07B72', linestyle='--', linewidth=1.3,
           label=f'True anomaly rate ({CONTAM*100:.2f}%)')
ax.set_xlabel(r'Threshold $\tau$')
ax.set_ylabel('Flagged (%)')
ax.set_title('Z-Score: Flagged Fraction vs. Threshold')
ax.legend(fontsize=9, frameon=False)

ax = axes[1]
ax.plot(k_vals, iqr_flagged, marker='s', color='#7BA7BC', linewidth=2)
ax.axhline(CONTAM * 100, color='#E07B72', linestyle='--', linewidth=1.3,
           label=f'True anomaly rate ({CONTAM*100:.2f}%)')
ax.set_xlabel('IQR multiplier k')
ax.set_ylabel('Flagged (%)')
ax.set_title('IQR Fence: Flagged Fraction vs. k')
ax.legend(fontsize=9, frameon=False)

plt.suptitle('Statistical Threshold Sweeps', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('plots/outlier_zscore_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_zscore_threshold_sweep.png')

In [ ]:
# ── Mahalanobis distance ───────────────────────────────────────────────────────────

mu      = X.mean(axis=0)
cov     = np.cov(X, rowvar=False)
cov_inv = np.linalg.pinv(cov)   # pseudo-inverse handles correlated features

diff     = X - mu
d2_mahal = np.einsum('ij,jk,ik->i', diff, cov_inv, diff)   # D_M^2
scores_mahal = d2_mahal

# Chi-squared threshold at alpha = contamination rate
threshold_mahal = chi2.ppf(1 - CONTAM, df=X.shape[1])
y_pred_mahal    = (d2_mahal > threshold_mahal).astype(int)

print(f'chi2_{X.shape[1]} threshold (alpha={CONTAM:.4f}): {threshold_mahal:.2f}')
print(f'Flagged: {y_pred_mahal.sum()} ({y_pred_mahal.mean()*100:.2f}%)')

from scipy.stats import chi2 as chi2_dist
fig, ax = plt.subplots(figsize=(8, 4))
x_chi = np.linspace(0, np.percentile(d2_mahal, 99), 300)
ax.hist(d2_mahal, bins=60, density=True, alpha=0.6, color='#4C8FAF', label='Observed $D_M^2$')
ax.plot(x_chi, chi2_dist.pdf(x_chi, df=X.shape[1]), color='#333333',
        linewidth=1.5, label=f'$\\chi^2_{{{X.shape[1]}}}$ density')
ax.axvline(threshold_mahal, color='#E07B72', linestyle='--', linewidth=1.3,
           label=f'Threshold ({threshold_mahal:.1f})')
ax.set_xlabel('$D_M^2$')
ax.set_ylabel('Density')
ax.set_title('Mahalanobis Distance: Observed vs. $\\chi^2_{36}$')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_mahal_chi2.png', dpi=150)
plt.show()
print('Saved plots/outlier_mahal_chi2.png')

In [ ]:
# ── Statistical methods scatter panel ─────────────────────────────────────────────

tau_zscore    = contam_threshold(max_z)
y_pred_zscore = (max_z > tau_zscore).astype(int)

scores_iqr  = ((X < Q1 - 1.5 * IQR) | (X > Q3 + 1.5 * IQR)).sum(axis=1).astype(float)
tau_iqr     = contam_threshold(scores_iqr)
y_pred_iqr  = (scores_iqr > tau_iqr).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (title, y_pred) in zip(axes, [
    (f'Z-Score (n={y_pred_zscore.sum()})',    y_pred_zscore),
    (f'IQR Fence (n={y_pred_iqr.sum()})',     y_pred_iqr),
    (f'Mahalanobis (n={y_pred_mahal.sum()})', y_pred_mahal),
]):
    outlier_scatter(ax, X_2d, y_pred, title)

handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=COLORS[l],
            markersize=8, label=LABELS[l]) for l in [0, 1]]
handles.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='none',
               markeredgecolor='red', markersize=8, label='Flagged'))
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Statistical Methods — PCA Projection', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('plots/outlier_stat_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_stat_scatter.png')

---
## 3. Distance-Based Detection: k-NN Outlier Score

The $k$th-nearest-neighbour distance $d_k(x_i)$ is a non-parametric local density estimate. Points in dense regions have small $d_k$; outliers in sparse regions have large $d_k$.

**Threshold:** We set the threshold using the contamination prior — flag the top `CONTAM` fraction of scores. The sorted k-distance plot is shown as a visual diagnostic.

**Choosing k:** Larger $k$ smooths the score estimate at the cost of locality. We sweep $k \in \{5, 10, 20, 50\}$ and report AUC for each.

In [ ]:
# ── k-NN distance sweep ───────────────────────────────────────────────────────────

k_values  = [5, 10, 20, 50]
knn_dists = {}

for k in k_values:
    knn = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree')
    knn.fit(X)
    dists, _ = knn.kneighbors(X)
    knn_dists[k] = dists[:, k]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for idx, k in enumerate(k_values):
    ax     = axes[idx]
    scores = knn_dists[k]
    d_k    = np.sort(scores)[::-1]
    tau    = contam_threshold(scores)
    n_flagged = (scores > tau).sum()
    auc    = roc_auc_score(y_out, scores)

    ax.plot(d_k, color='#4C8FAF', linewidth=1.2)
    ax.axhline(tau, color='#E07B72', linestyle='--', linewidth=1.2,
               label=f'tau={tau:.2f} (n={n_flagged})')
    ax.set_title(f'k = {k}  (AUC={auc:.3f})', fontsize=10)
    ax.set_xlabel('Sample rank', fontsize=9)
    ax.set_ylabel(f'{k}th-NN distance', fontsize=9)
    ax.legend(fontsize=8, frameon=False)

plt.suptitle('k-NN Distance Plots — Sweep of k (contamination-prior threshold)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('plots/outlier_knn_kdist_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_knn_kdist_sweep.png')

In [ ]:
# ── k-NN final model (k=10) ───────────────────────────────────────────────────

k_final       = 10
scores_knn    = knn_dists[k_final]
knn_threshold = contam_threshold(scores_knn)
y_pred_knn    = (scores_knn > knn_threshold).astype(int)

print(f'k-NN threshold (k={k_final}): {knn_threshold:.4f}')
print(f'Flagged: {y_pred_knn.sum()} ({y_pred_knn.mean()*100:.2f}%)')

fig, ax = plt.subplots(figsize=(6, 5))
outlier_scatter(ax, X_2d, y_pred_knn,
                f'k-NN Distance (k={k_final}, n flagged={y_pred_knn.sum()})')
plt.tight_layout()
plt.savefig('plots/outlier_knn_scatter.png', dpi=150)
plt.show()
print('Saved plots/outlier_knn_scatter.png')

---
## 4. Isolation Forest

Isolation Forest builds random trees that partition the feature space. Outliers are isolated in fewer splits than inliers. The anomaly score is:

$$s(x, n) = 2^{-\mathbb{E}[h(x)]/c(n)}, \quad c(n) = 2H(n-1) - \frac{2(n-1)}{n}$$

Score near **1** → outlier. Score near **0.5** → ambiguous. Score near **0** → inlier.

**Key hyperparameter:** `contamination` sets the decision threshold. We sweep it and show the score histogram to check for bimodal separation.

In [ ]:
# ── Isolation Forest: score distribution ──────────────────────────────────────

iforest_base  = IsolationForest(n_estimators=200, contamination='auto', random_state=SEED)
iforest_base.fit(X)
scores_if_raw = -iforest_base.score_samples(X)

fig, ax = plt.subplots(figsize=(8, 4))
for label in [0, 1]:
    mask = y == label
    ax.hist(scores_if_raw[mask], bins=40, alpha=0.65, density=True,
            color=COLORS[label], label=LABELS[label])
ax.set_xlabel('Isolation Forest anomaly score (higher = more anomalous)')
ax.set_ylabel('Density')
ax.set_title('Isolation Forest: Score Distribution by True Class')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_iforest_score_hist.png', dpi=150)
plt.show()
print('Saved plots/outlier_iforest_score_hist.png')

In [ ]:
# ── Contamination sweep ─────────────────────────────────────────────────────────

contam_vals    = [0.005, 0.01, 0.02, 0.05, 0.10]
contam_f1      = []
contam_flagged = []

for c in contam_vals:
    clf = IsolationForest(n_estimators=200, contamination=c, random_state=SEED)
    clf.fit(X)
    pred_if = (clf.predict(X) == -1).astype(int)
    contam_f1.append(f1_score(y_out, pred_if, zero_division=0))
    contam_flagged.append(pred_if.mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(contam_vals, contam_flagged, marker='o', color='#4C8FAF', linewidth=2)
ax.axhline(CONTAM * 100, color='#E07B72', linestyle='--', linewidth=1.2,
           label=f'True rate ({CONTAM*100:.2f}%)')
ax.set_xlabel('contamination')
ax.set_ylabel('Flagged (%)')
ax.set_title('Flagged Fraction vs. contamination')
ax.legend(fontsize=9, frameon=False)

ax = axes[1]
ax.plot(contam_vals, contam_f1, marker='s', color='#7BA7BC', linewidth=2)
ax.set_xlabel('contamination')
ax.set_ylabel('F1 (evaluated against true labels)')
ax.set_title('F1 Score vs. contamination')
ax.set_ylim(0, 1)

plt.suptitle('Isolation Forest: Contamination Sweep', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('plots/outlier_iforest_contamination_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_iforest_contamination_sweep.png')

In [ ]:
# ── Isolation Forest: final model ────────────────────────────────────────────────

iforest        = IsolationForest(n_estimators=200, contamination=CONTAM, random_state=SEED)
iforest.fit(X)
y_pred_iforest = (iforest.predict(X) == -1).astype(int)
scores_iforest = -iforest.score_samples(X)

print(f'Flagged: {y_pred_iforest.sum()} ({y_pred_iforest.mean()*100:.2f}%)')

fig, ax = plt.subplots(figsize=(6, 5))
outlier_scatter(ax, X_2d, y_pred_iforest,
                f'Isolation Forest (n flagged={y_pred_iforest.sum()})')
plt.tight_layout()
plt.savefig('plots/outlier_iforest_scatter.png', dpi=150)
plt.show()
print('Saved plots/outlier_iforest_scatter.png')

---
## 5. One-Class SVM

One-Class SVM learns a boundary around the inlier class in kernel feature space. The parameter $\nu \in (0,1]$ is an upper bound on the training outlier fraction and a lower bound on the support vector fraction:

$$\min_{w,\xi,\rho} \; \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_i \xi_i - \rho$$

We use the RBF kernel with `gamma='scale'`.

In [ ]:
# ── OC-SVM: nu sweep ───────────────────────────────────────────────────────────

nu_vals       = [0.005, 0.01, 0.02, 0.05, 0.10]
ocsvm_flagged = []
ocsvm_f1      = []

for nu in nu_vals:
    clf = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
    clf.fit(X)
    pred = (clf.predict(X) == -1).astype(int)
    ocsvm_flagged.append(pred.mean() * 100)
    ocsvm_f1.append(f1_score(y_out, pred, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x_pos = range(len(nu_vals))

ax = axes[0]
ax.bar(x_pos, ocsvm_flagged, color='#7BA7BC', alpha=0.8)
ax.axhline(CONTAM * 100, color='#E07B72', linestyle='--', linewidth=1.3,
           label=f'True rate ({CONTAM*100:.2f}%)')
ax.set_xticks(list(x_pos))
ax.set_xticklabels([str(v) for v in nu_vals])
ax.set_xlabel('nu')
ax.set_ylabel('Flagged (%)')
ax.set_title('OC-SVM: Flagged Fraction vs. nu')
ax.legend(fontsize=9, frameon=False)

ax = axes[1]
ax.bar(x_pos, ocsvm_f1, color='#4C8FAF', alpha=0.8)
ax.set_xticks(list(x_pos))
ax.set_xticklabels([str(v) for v in nu_vals])
ax.set_xlabel('nu')
ax.set_ylabel('F1 (evaluated against true labels)')
ax.set_title('OC-SVM: F1 vs. nu')
ax.set_ylim(0, 1)

plt.suptitle('One-Class SVM: nu Sweep', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('plots/outlier_ocsvm_nu_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_ocsvm_nu_sweep.png')

In [ ]:
# ── OC-SVM: final model, score histogram and scatter ─────────────────────────

ocsvm        = OneClassSVM(nu=CONTAM, kernel='rbf', gamma='scale')
ocsvm.fit(X)
y_pred_ocsvm = (ocsvm.predict(X) == -1).astype(int)
scores_ocsvm = -ocsvm.score_samples(X)

print(f'Flagged: {y_pred_ocsvm.sum()} ({y_pred_ocsvm.mean()*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for label in [0, 1]:
    mask = y == label
    ax.hist(scores_ocsvm[mask], bins=40, alpha=0.65, density=True,
            color=COLORS[label], label=LABELS[label])
ax.set_xlabel('Decision function distance (higher = more anomalous)')
ax.set_ylabel('Density')
ax.set_title('OC-SVM: Score Distribution by True Class')
ax.legend(frameon=False)

outlier_scatter(axes[1], X_2d, y_pred_ocsvm,
                f'OC-SVM (n flagged={y_pred_ocsvm.sum()})')

plt.tight_layout()
plt.savefig('plots/outlier_ocsvm_score_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_ocsvm_score_hist.png')

---
## 6. Gaussian Mixture Model

A GMM fits a probabilistic model to the data. The outlier score is the **negative log-likelihood**:

$$\text{score}(x_i) = -\log p(x_i) = -\log \sum_{k=1}^K \pi_k \, \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

Parameters are estimated by EM. We select $K$ by minimising BIC.

In [ ]:
# ── GMM: BIC/AIC sweep ────────────────────────────────────────────────────────

n_components_range = [1, 2, 3, 4, 5, 8, 10]
bic_vals = []
aic_vals = []

for K in n_components_range:
    gmm = GaussianMixture(n_components=K, covariance_type='full',
                          random_state=SEED, max_iter=200)
    gmm.fit(X)
    bic_vals.append(gmm.bic(X))
    aic_vals.append(gmm.aic(X))

best_K = n_components_range[np.argmin(bic_vals)]
print(f'BIC-optimal K: {best_K}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_components_range, bic_vals, marker='o', color='#4C8FAF', linewidth=2, label='BIC')
ax.plot(n_components_range, aic_vals, marker='s', color='#7BA7BC', linewidth=2,
        linestyle='--', label='AIC')
ax.axvline(best_K, color='#E07B72', linestyle=':', linewidth=1.5,
           label=f'BIC minimum (K={best_K})')
ax.set_xlabel('Number of components K')
ax.set_ylabel('Criterion value')
ax.set_title('GMM: BIC and AIC vs. K')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_gmm_bic_aic.png', dpi=150)
plt.show()
print('Saved plots/outlier_gmm_bic_aic.png')

In [ ]:
# ── GMM: final model, score histogram and scatter ────────────────────────────

gmm_final = GaussianMixture(n_components=best_K, covariance_type='full',
                             random_state=SEED, max_iter=200)
gmm_final.fit(X)

scores_gmm = -gmm_final.score_samples(X)
tau_gmm    = contam_threshold(scores_gmm)
y_pred_gmm = (scores_gmm > tau_gmm).astype(int)

print(f'GMM threshold (-log p): {tau_gmm:.2f}')
print(f'Flagged: {y_pred_gmm.sum()} ({y_pred_gmm.mean()*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for label in [0, 1]:
    mask = y == label
    ax.hist(scores_gmm[mask], bins=40, alpha=0.65, density=True,
            color=COLORS[label], label=LABELS[label])
ax.axvline(tau_gmm, color='#333333', linestyle='--', linewidth=1.3,
           label=f'Threshold ({tau_gmm:.2f})')
ax.set_xlabel('Negative log-likelihood')
ax.set_ylabel('Density')
ax.set_title(f'GMM Score Distribution (K={best_K})')
ax.legend(frameon=False)

outlier_scatter(axes[1], X_2d, y_pred_gmm,
                f'GMM (K={best_K}, n flagged={y_pred_gmm.sum()})')

plt.tight_layout()
plt.savefig('plots/outlier_gmm_score_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_gmm_score_hist.png')

---
## 7. Variational Autoencoder

A VAE learns a compressed latent representation by maximising the ELBO:

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{z \sim q_\phi(z|x)}[\log p_\theta(x|z)] - D_{\text{KL}}(q_\phi(z|x) \| p(z))$$

After training, the **reconstruction error** serves as the anomaly score. The reparameterization trick allows backpropagation through sampling:

$$\varepsilon \sim \mathcal{N}(0,I), \qquad z = \mu + \sigma \odot \varepsilon$$

In [ ]:
# ── VAE architecture ───────────────────────────────────────────────────────────

INPUT_DIM  = X.shape[1]   # 36
HIDDEN_DIM = 128

class Encoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=8):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, 64)
        self.mu   = nn.Linear(64, latent_dim)
        self.logv = nn.Linear(64, latent_dim)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        return self.mu(h), self.logv(h)


class Decoder(nn.Module):
    def __init__(self, latent_dim=8, hidden_dim=HIDDEN_DIM, output_dim=INPUT_DIM):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 64)
        self.fc2 = nn.Linear(64, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = torch.relu(self.fc1(z))
        h = torch.relu(self.fc2(h))
        return self.out(h)


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=8):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z    = self.reparameterize(mu, logvar)
        xhat = self.decoder(z)
        return xhat, mu, logvar


def elbo_loss(x, xhat, mu, logvar, beta=1.0):
    recon = nn.functional.mse_loss(xhat, x, reduction='sum')
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


def get_reconstruction_errors(model, X_tensor):
    model.eval()
    with torch.no_grad():
        mu, _ = model.encoder(X_tensor)
        xhat  = model.decoder(mu)
        errs  = ((xhat - X_tensor) ** 2).mean(dim=1)
    return errs.numpy()


print('VAE architecture defined.')

In [ ]:
# ── Latent dim sweep ───────────────────────────────────────────────────────────

X_tensor = torch.tensor(X, dtype=torch.float32)

rng       = np.random.default_rng(SEED)
idx       = rng.permutation(len(X))
n_train   = int(0.8 * len(X))
train_idx, val_idx = idx[:n_train], idx[n_train:]

X_train_t    = X_tensor[train_idx]
X_val_t      = X_tensor[val_idx]
train_loader = DataLoader(TensorDataset(X_train_t), batch_size=64, shuffle=True)

latent_dims         = [2, 4, 8, 16]
SWEEP_EPOCHS        = 150
val_recon_histories = {}

for ld in latent_dims:
    torch.manual_seed(SEED)
    model  = VAE(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=ld)
    optim_ = optim.Adam(model.parameters(), lr=1e-3)
    val_history = []

    for epoch in range(SWEEP_EPOCHS):
        model.train()
        for (xb,) in train_loader:
            optim_.zero_grad()
            xhat, mu, logvar = model(xb)
            loss, _, _ = elbo_loss(xb, xhat, mu, logvar)
            loss.backward()
            optim_.step()

        model.eval()
        with torch.no_grad():
            xhat_val, _, _ = model(X_val_t)
            val_recon = nn.functional.mse_loss(xhat_val, X_val_t, reduction='mean').item()
        val_history.append(val_recon)

    val_recon_histories[ld] = val_history
    print(f'latent_dim={ld:>2}  final val recon MSE: {val_history[-1]:.4f}')

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
palette = ['#4C8FAF', '#7BA7BC', '#E07B72', '#A3C48B']

for ax, ld, color in zip(axes.flatten(), latent_dims, palette):
    ax.plot(val_recon_histories[ld], color=color, linewidth=1.5)
    ax.set_title(f'latent_dim = {ld}', fontsize=10)
    ax.set_xlabel('Epoch', fontsize=9)
    ax.set_ylabel('Val reconstruction MSE', fontsize=9)

plt.suptitle('VAE: Latent Dimension Sweep — Validation Reconstruction Loss', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('plots/outlier_vae_latent_dim_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_vae_latent_dim_sweep.png')

In [ ]:
# ── Final VAE training (300 epochs) ─────────────────────────────────────────────

FINAL_EPOCHS = 300
LATENT_DIM   = 8

torch.manual_seed(SEED)
vae_final = VAE(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM)
optimizer = optim.Adam(vae_final.parameters(), lr=1e-3)

history = {'total': [], 'recon': [], 'kl': []}

for epoch in range(FINAL_EPOCHS):
    vae_final.train()
    epoch_total = epoch_recon = epoch_kl = 0.0
    for (xb,) in train_loader:
        optimizer.zero_grad()
        xhat, mu, logvar = vae_final(xb)
        loss, recon, kl = elbo_loss(xb, xhat, mu, logvar)
        loss.backward()
        optimizer.step()
        epoch_total += loss.item()
        epoch_recon += recon.item()
        epoch_kl    += kl.item()

    n_batches = len(train_loader)
    history['total'].append(epoch_total / n_batches)
    history['recon'].append(epoch_recon / n_batches)
    history['kl'].append(epoch_kl / n_batches)

print(f'Final training loss: {history["total"][-1]:.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['total'], label='Total ELBO loss', color='#333333', linewidth=1.5)
ax.plot(history['recon'], label='Reconstruction',  color='#4C8FAF', linewidth=1.5, linestyle='--')
ax.plot(history['kl'],    label='KL divergence',   color='#E07B72', linewidth=1.5, linestyle=':')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (per batch, summed)')
ax.set_title(f'VAE Training Curves (latent_dim={LATENT_DIM})')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_vae_training_curves.png', dpi=150)
plt.show()
print('Saved plots/outlier_vae_training_curves.png')

In [ ]:
# ── VAE reconstruction error scores, scatter, latent space ──────────────────────

scores_vae    = get_reconstruction_errors(vae_final, X_tensor)
vae_threshold = contam_threshold(scores_vae)
y_pred_vae    = (scores_vae > vae_threshold).astype(int)

print(f'VAE recon error threshold: {vae_threshold:.4f}')
print(f'Flagged: {y_pred_vae.sum()} ({y_pred_vae.mean()*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for label in [0, 1]:
    mask = y == label
    ax.hist(scores_vae[mask], bins=40, alpha=0.65, density=True,
            color=COLORS[label], label=LABELS[label])
ax.axvline(vae_threshold, color='#333333', linestyle='--', linewidth=1.3,
           label=f'Threshold ({vae_threshold:.3f})')
ax.set_xlabel('Reconstruction MSE')
ax.set_ylabel('Density')
ax.set_title(f'VAE: Reconstruction Error Distribution (latent_dim={LATENT_DIM})')
ax.legend(frameon=False)

outlier_scatter(axes[1], X_2d, y_pred_vae,
                f'VAE (latent_dim={LATENT_DIM}, n flagged={y_pred_vae.sum()})')

plt.tight_layout()
plt.savefig('plots/outlier_vae_score_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_vae_score_hist.png')

# ── 2D latent space visualization ────────────────────────────────────────────────
torch.manual_seed(SEED)
vae_2d     = VAE(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=2)
optimizer2 = optim.Adam(vae_2d.parameters(), lr=1e-3)

for epoch in range(300):
    vae_2d.train()
    for (xb,) in train_loader:
        optimizer2.zero_grad()
        xhat, mu, logvar = vae_2d(xb)
        loss, _, _ = elbo_loss(xb, xhat, mu, logvar)
        loss.backward()
        optimizer2.step()

vae_2d.eval()
with torch.no_grad():
    mu_all, _ = vae_2d.encoder(X_tensor)
Z_latent = mu_all.numpy()

fig, ax = plt.subplots(figsize=(6, 5))
for label in [0, 1]:
    mask = y == label
    ax.scatter(Z_latent[mask, 0], Z_latent[mask, 1],
               c=COLORS[label], label=LABELS[label],
               s=10, alpha=0.7, linewidths=0)
ax.set_xlabel('Latent dim 1')
ax.set_ylabel('Latent dim 2')
ax.set_title('VAE Latent Space (latent_dim=2, colored by true label)')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_vae_latent_space.png', dpi=150)
plt.show()
print('Saved plots/outlier_vae_latent_space.png')

---
## 8. Evaluation and Comparison

Now we reveal the labels and evaluate all methods head-to-head.

All binary predictions flag the top `CONTAM` fraction of scores. ROC-AUC uses the continuous anomaly score and is threshold-independent.

In [ ]:
# ── Compute metrics for all methods ───────────────────────────────────────────

methods_eval = {
    'Z-Score':          (max_z,           y_pred_zscore),
    'IQR':              (scores_iqr,       y_pred_iqr),
    'Mahalanobis':      (scores_mahal,     y_pred_mahal),
    'k-NN (k=10)':      (scores_knn,       y_pred_knn),
    'Isolation Forest': (scores_iforest,   y_pred_iforest),
    'One-Class SVM':    (scores_ocsvm,     y_pred_ocsvm),
    'GMM':              (scores_gmm,       y_pred_gmm),
    'VAE':              (scores_vae,       y_pred_vae),
}

results = {name: compute_metrics(y_out, preds, scores)
           for name, (scores, preds) in methods_eval.items()}

df_results = pd.DataFrame(results).T.round(3)
print(df_results.to_string())

In [ ]:
# ── ROC curve overlay ─────────────────────────────────────────────────────────

roc_palette = ['#E07B72', '#7BA7BC', '#4C8FAF', '#A3C48B', '#F4A460', '#9370DB', '#333333', '#888888']

fig, ax = plt.subplots(figsize=(7, 6))
for (name, (scores, _)), color in zip(methods_eval.items(), roc_palette):
    fpr, tpr, _ = roc_curve(y_out, scores)
    auc_val     = roc_auc_score(y_out, scores)
    ax.plot(fpr, tpr, color=color, linewidth=1.8,
            label=f'{name} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.9, alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: All Outlier Detection Methods')
ax.legend(fontsize=8, frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig('plots/outlier_roc_comparison.png', dpi=150)
plt.show()
print('Saved plots/outlier_roc_comparison.png')

In [ ]:
# ── Grouped bar chart: Precision / Recall / F1 ────────────────────────────────

method_names   = list(results.keys())
precision_vals = [results[m]['Precision'] for m in method_names]
recall_vals    = [results[m]['Recall']    for m in method_names]
f1_vals        = [results[m]['F1']        for m in method_names]

x     = np.arange(len(method_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, precision_vals, width, label='Precision', color='#4C8FAF', alpha=0.85)
ax.bar(x,          recall_vals,    width, label='Recall',    color='#E07B72', alpha=0.85)
ax.bar(x + width,  f1_vals,        width, label='F1',        color='#A3C48B', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(method_names, rotation=20, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Outlier Detection: Precision / Recall / F1 Comparison')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('plots/outlier_metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_metrics_bar.png')

In [ ]:
# ── All methods scatter panel ──────────────────────────────────────────────────

panel_methods = [
    ('Z-Score',          y_pred_zscore),
    ('Mahalanobis',      y_pred_mahal),
    ('k-NN (k=10)',      y_pred_knn),
    ('Isolation Forest', y_pred_iforest),
    ('GMM',              y_pred_gmm),
    ('VAE',              y_pred_vae),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (name, y_pred) in zip(axes, panel_methods):
    f1 = results.get(name, {}).get('F1', 0.0)
    outlier_scatter(ax, X_2d, y_pred, f'{name}\n(n={y_pred.sum()}, F1={f1:.2f})')

handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=COLORS[l],
                  markersize=8, label=LABELS[l]) for l in [0, 1]]
handles.append(Line2D([0],[0], marker='o', color='w', markerfacecolor='none',
               markeredgecolor='red', markersize=8, label='Flagged'))
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.03))
plt.suptitle('All Methods — PCA Projection with Flagged Outliers', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('plots/outlier_all_methods_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/outlier_all_methods_scatter.png')